<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

# Questão 1

Utilize o dataset Iris disponível no scikit-learn.
Divida os dados em treino e teste utilizando divisão estratificada.

**Solução**:

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# 1. Carregar o dataset Iris
iris = load_iris()
X = iris.data
y = iris.target

# 2. Dividir em treino e teste com divisão estratificada
# O parâmetro stratify=y garante a proporção das classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Verificação das dimensões
print(f"Tamanho do treino: {X_train.shape[0]} amostras")
print(f"Tamanho do teste: {X_test.shape[0]} amostras")

# Questão 2

Treine um modelo utilizando `DecisionTreeClassifier`.

Depois calcule:

- acurácia no treino
- acurácia no teste

**Solução**:

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Instanciar o modelo
# Usamos random_state para garantir que o resultado seja reproduzível
clf = DecisionTreeClassifier(random_state=42)

# 2. Treinar o modelo
clf.fit(X_train, y_train)

# 3. Fazer predições para ambos os conjuntos
y_pred_train = clf.predict(X_train)
y_pred_test = clf.predict(X_test)

# 4. Calcular a acurácia
acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f"Acurácia no Treino: {acc_train:.2%}")
print(f"Acurácia no Teste: {acc_test:.2%}")

# Questão 3

Utilize `plot_tree()` para visualizar a árvore treinada.

Responda:

1. Qual atributo aparece na raiz?
2. Qual é a profundidade da árvore?

**Solução**:

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# Configurar o tamanho da imagem para melhor visualização
plt.figure(figsize=(15, 10))

# Plotar a árvore
plot_tree(clf, 
          feature_names=iris.feature_names, 
          class_names=iris.target_names, 
          filled=True, 
          rounded=True)

plt.show()

print(f"Profundidade da árvore: {clf.get_depth()}")

**1. Qual atributo aparece na raiz?**
O atributo presente no nó raiz é o petal length (cm) (comprimento da pétala). No nó raiz, a árvore decidiu que o corte em petal length (cm) <= 2.45 era o que mais reduzia a impureza do conjunto de dados. Vale ressaltar que esse único corte foi capaz de separar perfeitamente as 35 amostras da classe Setosa (nó folha à esquerda com Gini = 0.0), deixando as outras 70 amostras (Versicolor e Virginica) para serem processadas nos níveis seguintes. O Ganho de Informação gerado por esse atributo foi o maior entre todas as features disponíveis no primeiro passo.

**2. Qual é a profundidade da árvore?**
A profundidade da árvore é 5. A profundidade é medida pelo caminho mais longo da raiz até uma folha. Seguindo pelo lado direito da árvore (onde os dados são mais "misturados"), podemos contar os níveis de decisão:

Raiz: petal length (cm) <= 2.45 (Nível 0)

Nó 1: petal width (cm) <= 1.55 (Nível 1)

Nó 2: petal width (cm) <= 1.7 (Nível 2)

Nó 3: sepal length (cm) <= 4.85 (Nível 3)

Nó 4: sepal width (cm) <= 3.00 (Nível 4)

As folhas que surgem a partir deste último nó estão no Nível 5, que é a profundidade máxima da árvore.

# Questão 4

Treine dez árvores com:

- max_depth = 1
- max_depth = 2
- max_depth = 3
...
- max_depth = 9
- max_depth = None

Registre em uma tabela para cada árvore:

- acurácia no treino
- acurácia no teste
- profundidade da árvore
- número de folhas

**Solução**:

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# 1. Carregar o dataset Iris
iris = load_iris()
X = iris.data
y = iris.target

# 2. Divisão estratificada (conforme a Questão 1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 3. Preparação para o loop
profundidades_para_testar = list(range(1, 10)) + [None]
lista_resultados = []

# 4. Treinar as dez árvores
for d in profundidades_para_testar:
    # Instanciar e treinar
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    
    # Obter métricas e informações da árvore
    y_pred_train = clf.predict(X_train)
    y_pred_test = clf.predict(X_test)
    
    acc_treino = accuracy_score(y_train, y_pred_train)
    acc_teste = accuracy_score(y_test, y_pred_test)
    profundidade_real = clf.get_depth()
    num_folhas = clf.get_n_leaves()
    
    # Armazenar os dados
    lista_resultados.append({
        "Max Depth Config": "None" if d is None else d,
        "Profundidade Real": profundidade_real,
        "Nº de Folhas": num_folhas,
        "Acurácia Treino": f"{acc_treino:.2%}",
        "Acurácia Teste": f"{acc_teste:.2%}"
    })

# 5. Criar o DataFrame e exibir a tabela
df_resultados = pd.DataFrame(lista_resultados)
print(df_resultados.to_string(index=False))

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

**Em qual profundidade começa o overfitting?**
O overfitting começa na profundidade 4. Isso é observado porque, a partir desse ponto, a acurácia no conjunto de treino continua subindo (chegando a 99%), enquanto a acurácia no conjunto de teste sofre uma queda significativa (caindo para 88%). Esse fenômeno indica que o modelo parou de aprender padrões gerais e começou a "decorar" ruídos ou casos muito específicos do treino que não se repetem nos novos dados.

**Por que a árvore consegue 100% no treino quando max_depth=None?**
A árvore atinge 100% de acurácia no treino quando max_depth=None porque o algoritmo não possui restrições para parar de crescer. Ele continua criando divisões e ramificações até que cada nó folha contenha amostras de apenas uma única classe, atingindo a pureza total (Gini = 0.0). Sem um limite de profundidade, a árvore acaba criando um "mapa de memória" perfeito do conjunto de treinamento, resultando em zero erros para esses dados específicos, embora isso prejudique sua capacidade de generalização.

# Questão 5

Treine dois modelos:

- criterion = "gini"
- criterion = "entropy"

Compare:

- profundidade da árvore
- acurácia

**Solução**:

In [ ]:
# 1. Instanciar e treinar o modelo com critério Gini
clf_gini = DecisionTreeClassifier(criterion="gini", random_state=42)
clf_gini.fit(X_train, y_train)

# 2. Instanciar e treinar o modelo com critério Entropia
clf_entropy = DecisionTreeClassifier(criterion="entropy", random_state=42)
clf_entropy.fit(X_train, y_train)

# 3. Coleta e comparação de métricas
resultados_comp = []
for nome, modelo in [("Gini", clf_gini), ("Entropy", clf_entropy)]:
    acc_teste = accuracy_score(y_test, modelo.predict(X_test))
    resultados_comp.append({
        "Critério": nome,
        "Profundidade": modelo.get_depth(),
        "Acurácia Teste": f"{acc_teste:.2%}"
    })

# Exibir comparação
df_comp = pd.DataFrame(resultados_comp)
print(df_comp.to_string(index=False))

# Questão 6

Escolha um hiperparâmetro e investigue seu impacto.

Sugestões:

- max_depth
- min_samples_split
- min_samples_leaf
- criterion

Mostre resultados e interprete.
- melhor modelo encontrado
- acurácia
- parâmetros

**Solução**:

In [ ]:
# Investigação de Hiperparâmetro (min_samples_leaf)

import pandas as pd
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import accuracy_score

# 1. Definir o intervalo de valores para o hiperparâmetro
valores_leaf = [1, 2, 3, 4, 5]
lista_resultados_estudo = []

# 2. Loop para treinar e avaliar os modelos
for msl in valores_leaf:
    # Instanciar o modelo com o parâmetro atual
    modelo_temp = DecisionTreeClassifier(min_samples_leaf=msl, random_state=42)
    modelo_temp.fit(X_train, y_train)
    
    # Fazer predições
    pred_treino = modelo_temp.predict(X_train)
    pred_teste = modelo_temp.predict(X_test)
    
    # Calcular métricas
    acc_treino = accuracy_score(y_train, pred_treino)
    acc_teste = accuracy_score(y_test, pred_teste)
    
    # Armazenar informações
    lista_resultados_estudo.append({
        "min_samples_leaf": msl,
        "Profundidade": modelo_temp.get_depth(),
        "Nº de Folhas": modelo_temp.get_n_leaves(),
        "Acurácia Treino": f"{acc_treino:.2%}",
        "Acurácia Teste": f"{acc_teste:.2%}"
    })

# 3. Exibir a tabela de comparação
df_estudo = pd.DataFrame(lista_resultados_estudo)
print("Tabela de Impacto do Hiperparâmetro min_samples_leaf:")
print(df_estudo.to_string(index=False))

# 4. Identificar o melhor modelo (baseado na acurácia de teste)
melhor_resultado = df_estudo.sort_values(by="Acurácia Teste", ascending=False).iloc[0]
print("\n--- Melhor Modelo Encontrado ---")
print(f"Parâmetro min_samples_leaf: {melhor_resultado['min_samples_leaf']}")
print(f"Acurácia alcançada: {melhor_resultado['Acurácia Teste']}")